# PCL Classifier — Training Pipeline

Binary classification: predict whether a news paragraph contains Patronizing and
Condescending Language (PCL) towards a vulnerable community.

**Evaluation metric:** F1 of the positive class only.

**Key ideas (from PALI-NLP, SemEval 2022 winner):**
- `deberta-v3-base` backbone (stronger than roberta-base, same size as roberta-large)
- Grouped Layer-wise Learning Rate Decay (Grouped LLRD)
- Weighted Random Sampler (WRS) to handle 9.5:1 class imbalance
- Cosine annealing scheduler with linear warmup
- Metadata (keyword + country) prepended to each paragraph
- Early stopping on validation F1, threshold tuning post-training

**Data split:**
- Official train (8,375) → stratified 90/10 → our **train** + **validation**
- Official dev (2,094) → our held-out **test** (labels available for local evaluation)
- Official test (3,832) → unlabelled; generate `test.txt` for leaderboard

In [ ]:
# !pip install transformers torch scikit-learn pandas numpy tqdm seaborn tiktoken

In [ ]:
import os, sys, html, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_cosine_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Add notebooks/ to path so we can import the data loader
sys.path.append(os.path.dirname(os.path.abspath("__file__")))
from dont_patronize_me import DontPatronizeMe

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def get_device():
    """Prefer CUDA > Apple MPS (GPU) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")  # Apple Silicon GPU
    return torch.device("cpu")

set_seed(42)
DEVICE = get_device()
print(f"Using device: {DEVICE}")

## 1. Data Loading and Preprocessing

We load all annotated paragraphs via the helper class, then filter to the
official train/dev splits using the provided par_id CSV files.

The official dev set (2,094 rows) is treated as our **held-out test** since
the official test set has no labels. We further split the official train into
our own 90/10 train/validation split (stratified by label).

In [ ]:
DATA_DIR = "../data"

# Load all 10,469 labelled paragraphs
dpm = DontPatronizeMe(DATA_DIR, os.path.join(DATA_DIR, "task4_test.tsv"))
dpm.load_task1()
full_df = dpm.train_task1_df.copy()
full_df["par_id"] = full_df["par_id"].astype(str)
full_df["orig_label"] = full_df["orig_label"].astype(int)

# Load official train/dev par_id lists
train_ids = pd.read_csv(os.path.join(DATA_DIR, "train_semeval_parids-labels.csv"))
dev_ids   = pd.read_csv(os.path.join(DATA_DIR, "dev_semeval_parids-labels.csv"))
train_ids["par_id"] = train_ids["par_id"].astype(str)
dev_ids["par_id"]   = dev_ids["par_id"].astype(str)

# Filter into official splits
train_pool = full_df[full_df["par_id"].isin(train_ids["par_id"])].reset_index(drop=True)
test_df    = full_df[full_df["par_id"].isin(dev_ids["par_id"])].reset_index(drop=True)

# Stratified 90/10 split of the train pool → our train + validation
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.10,
    stratify=train_pool["label"],
    random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} rows  |  PCL: {train_df['label'].sum()} ({train_df['label'].mean():.1%})")
print(f"Val:   {len(val_df)} rows   |  PCL: {val_df['label'].sum()} ({val_df['label'].mean():.1%})")
print(f"Test:  {len(test_df)} rows  |  PCL: {test_df['label'].sum()} ({test_df['label'].mean():.1%})")

In [ ]:
def build_input_text(row):
    """
    Prepend keyword and country metadata as special tokens before the paragraph.
    This gives the model free context about which vulnerable group and which
    national media culture the paragraph comes from — an idea from the
    PALI-NLP winning system.
    """
    text = html.unescape(row["text"])  # fix &amp; &lt; etc. (65 occurrences in train)
    return f"keyword: {row['keyword']} | country: {row['country']} | {text}"


for df in [train_df, val_df, test_df]:
    df["input_text"] = df.apply(build_input_text, axis=1)

# Load unlabelled test set for leaderboard submission
dpm.load_test()
submit_df = dpm.test_set_df.copy()
submit_df["input_text"] = submit_df.apply(build_input_text, axis=1)

print("Sample input:")
print(train_df["input_text"].iloc[0])

## 2. Dataset and DataLoader

A minimal PyTorch Dataset tokenizes text on the fly.
For training we attach a **Weighted Random Sampler (WRS)** to the DataLoader.

**WRS formula (PALI-NLP):**  
Each sample weight = `1 / √(class_ratio)` where `class_ratio` is the fraction
of that class in the training set. The square-root dampens the correction,
making it softer than full inverse-frequency weighting.

In [ ]:
class PCLDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels  # None for unlabelled inference

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {"text": self.texts[idx]}
        if self.labels is not None:
            item["label"] = self.labels[idx]
        return item

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
MODEL_NAME  = "microsoft/deberta-v3-base"  # swap for "roberta-large" if preferred
MAX_LENGTH  = 250    # covers 99th percentile of token lengths (from EDA)
BATCH_SIZE  = 8      # reduced for MPS; effective batch = BATCH_SIZE × GRAD_ACCUM
GRAD_ACCUM  = 2      # gradient accumulation (emulates batch size 16)
# ──────────────────────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def collate_fn(batch):
    """Tokenize a batch and pad to the longest sequence in the batch."""
    texts  = [item["text"] for item in batch]
    labels = [item["label"] for item in batch] if "label" in batch[0] else None

    encoding = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    if labels is not None:
        encoding["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoding


# ── Weighted Random Sampler ────────────────────────────────────────────────────
labels_array = train_df["label"].values
n_pos = labels_array.sum()
n_neg = len(labels_array) - n_pos
kappa_pos = n_pos / len(labels_array)
kappa_neg = n_neg / len(labels_array)

w_pos = 1.0 / np.sqrt(kappa_pos)  # ≈ 3.25
w_neg = 1.0 / np.sqrt(kappa_neg)  # ≈ 1.05
sample_weights = np.where(labels_array == 1, w_pos, w_neg)

sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_df),
    replacement=True
)
print(f"WRS weights — positive: {w_pos:.3f}, negative: {w_neg:.3f}")

# ── DataLoaders ────────────────────────────────────────────────────────────────
train_dataset = PCLDataset(train_df["input_text"].tolist(), train_df["label"].tolist())
val_dataset   = PCLDataset(val_df["input_text"].tolist(),   val_df["label"].tolist())
test_dataset  = PCLDataset(test_df["input_text"].tolist(),  test_df["label"].tolist())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,   collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate_fn)

## 3. Model and Optimizer

We use **Grouped Layer-wise Learning Rate Decay (Grouped LLRD)** from PALI-NLP.

The transformer layers are split into 3 groups from bottom to top, with
learning rates: `η/λ`, `η`, `η·λ` respectively.  
Intuition: bottom layers encode general syntactic features that should change
slowly; top layers encode task-specific features and can adapt faster.

**Optimal values (PALI-NLP ablation):** η = 1e-5, λ = 1.6

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
BASE_LR     = 1e-5
LAMBDA      = 1.6    # PALI-NLP optimal for Subtask 1
N_GROUPS    = 3
WEIGHT_DECAY = 0.01
DROPOUT     = 0.4    # passed via config; DeBERTa uses it in the classifier head
N_EPOCHS    = 10
WARMUP_RATIO = 0.10  # fraction of total steps used for linear warmup
PATIENCE    = 3      # early stopping: stop after N epochs without val F1 improvement
# ──────────────────────────────────────────────────────────────────────────────

def get_grouped_llrd_params(model, base_lr, lambda_val, n_groups=3):
    """
    Returns optimizer parameter groups with Grouped LLRD.

    Encoder layers are divided into n_groups from bottom (lowest LR) to top
    (highest LR). Embeddings join the bottom group; pooler and classifier join
    the top group.

    LR schedule: group g gets base_lr * lambda_val^(g - 1) for g in {0,1,2},
    which gives [base_lr/λ, base_lr, base_lr·λ] for groups [0, 1, 2].
    """
    if hasattr(model, "deberta"):
        encoder_layers = model.deberta.encoder.layer
        embeddings     = model.deberta.embeddings
        backbone       = "deberta"
    elif hasattr(model, "roberta"):
        encoder_layers = model.roberta.encoder.layer
        embeddings     = model.roberta.embeddings
        backbone       = "roberta"
    else:
        return [{"params": model.parameters(), "lr": base_lr}]

    n_layers   = len(encoder_layers)
    group_size = n_layers // n_groups

    # Layer index → learning rate
    def layer_lr(idx):
        group_idx = min(idx // group_size, n_groups - 1)
        return base_lr * (lambda_val ** (group_idx - 1))

    param_groups = []
    seen = set()

    def add_group(params_iter, lr):
        params = [p for p in params_iter if id(p) not in seen and p.requires_grad]
        for p in params:
            seen.add(id(p))
        if params:
            param_groups.append({"params": params, "lr": lr, "weight_decay": WEIGHT_DECAY})

    # Embeddings → bottom LR
    add_group(embeddings.parameters(), base_lr / lambda_val)

    # Encoder layers
    for i, layer in enumerate(encoder_layers):
        add_group(layer.parameters(), layer_lr(i))

    # DeBERTa-specific encoder-level params (relative embeddings, final LayerNorm)
    if backbone == "deberta":
        top_lr = base_lr * lambda_val
        for attr in ("rel_embeddings", "LayerNorm"):
            if hasattr(model.deberta.encoder, attr):
                add_group(getattr(model.deberta.encoder, attr).parameters(), top_lr)

    # Pooler and classifier → top LR (most task-specific)
    top_lr = base_lr * lambda_val
    if hasattr(model, "pooler") and model.pooler is not None:
        add_group(model.pooler.parameters(), top_lr)
    if hasattr(model, "classifier"):
        add_group(model.classifier.parameters(), top_lr)

    total_params = sum(p.numel() for g in param_groups for p in g["params"])
    print(f"Grouped LLRD: {len(param_groups)} param groups | {total_params:,} trainable params")
    for i, g in enumerate(param_groups):
        n = sum(p.numel() for p in g["params"])
        print(f"  Group {i}: lr={g['lr']:.2e}  params={n:,}")

    return param_groups


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT,
    ignore_mismatched_sizes=True
)
model.gradient_checkpointing_enable()  # reduces memory for MPS
model.to(DEVICE)

param_groups = get_grouped_llrd_params(model, BASE_LR, LAMBDA, N_GROUPS)
optimizer    = AdamW(param_groups)

# Total training steps for the scheduler
total_steps  = (len(train_loader) // GRAD_ACCUM) * N_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"\nTotal steps: {total_steps} | Warmup steps: {warmup_steps}")

## 4. Training

Training loop with:
- Gradient accumulation (effective batch = BATCH_SIZE × GRAD_ACCUM)
- Gradient clipping (max norm 1.0) for stability
- Early stopping on validation F1 (patience = 3 epochs)
- Best model checkpoint saved in-memory and written to `../BestModel/`

In [ ]:
def evaluate(model, loader, threshold=0.5):
    """Return predictions, probabilities, and F1 against a given threshold."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").numpy()
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            probs  = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels)
    preds = (np.array(all_probs) >= threshold).astype(int)
    f1    = f1_score(all_labels, preds, pos_label=1, zero_division=0)
    return np.array(all_probs), np.array(all_labels), f1


best_val_f1   = 0.0
best_state    = None
patience_ctr  = 0
history       = {"train_loss": [], "val_f1": []}

for epoch in range(1, N_EPOCHS + 1):
    # ── Training ────────────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    steps      = 0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{N_EPOCHS}")):
        labels = batch.pop("labels").to(DEVICE)
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        loss   = model(**batch, labels=labels).loss / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            steps += 1

    avg_loss = total_loss / len(train_loader)

    # ── Validation ──────────────────────────────────────────────────────────────
    _, _, val_f1 = evaluate(model, val_loader)
    history["train_loss"].append(avg_loss)
    history["val_f1"].append(val_f1)
    print(f"Epoch {epoch}  |  loss: {avg_loss:.4f}  |  val F1: {val_f1:.4f}")

    # ── Early stopping ──────────────────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
        print(f"  ✓ New best val F1: {best_val_f1:.4f}")
    else:
        patience_ctr += 1
        print(f"  No improvement. Patience: {patience_ctr}/{PATIENCE}")
        if patience_ctr >= PATIENCE:
            print("Early stopping triggered.")
            break

# Restore best checkpoint
model.load_state_dict(best_state)
model.to(DEVICE)

# Persist best model for the BestModel/ submission folder
os.makedirs("../BestModel", exist_ok=True)
model.save_pretrained("../BestModel")
tokenizer.save_pretrained("../BestModel")
print(f"\nBest model saved to ../BestModel  (val F1 = {best_val_f1:.4f})")

## 5. Threshold Tuning

The default decision threshold is 0.5, but given the class imbalance the
optimal F1 threshold on the validation set is usually lower (around 0.3–0.4).
We sweep thresholds and pick the one that maximises validation F1.

In [ ]:
val_probs, val_labels, _ = evaluate(model, val_loader)

thresholds = np.arange(0.05, 0.95, 0.01)
f1_scores  = [f1_score(val_labels, (val_probs >= t).astype(int), pos_label=1, zero_division=0)
              for t in thresholds]

BEST_THRESHOLD = thresholds[np.argmax(f1_scores)]
best_f1_at_t   = max(f1_scores)

plt.figure(figsize=(8, 3))
plt.plot(thresholds, f1_scores)
plt.axvline(BEST_THRESHOLD, color="red", linestyle="--", label=f"Best t={BEST_THRESHOLD:.2f}  F1={best_f1_at_t:.4f}")
plt.xlabel("Threshold")
plt.ylabel("Val F1 (positive class)")
plt.title("Threshold vs. Validation F1")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Best threshold: {BEST_THRESHOLD:.2f}  →  Val F1: {best_f1_at_t:.4f}")

## 6. Evaluation on Held-out Test (Official Dev Set)

We now evaluate the tuned model on the official dev set — the closest proxy
we have to unseen data, since the official test labels are unavailable.

In [ ]:
test_probs, test_labels, test_f1_default = evaluate(model, test_loader, threshold=0.5)
_, _, test_f1_tuned = evaluate(model, test_loader, threshold=BEST_THRESHOLD)
test_preds = (test_probs >= BEST_THRESHOLD).astype(int)

print(f"Test F1 (t=0.50):               {test_f1_default:.4f}")
print(f"Test F1 (t={BEST_THRESHOLD:.2f}, tuned):  {test_f1_tuned:.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=["No PCL", "PCL"]))

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: No PCL", "Pred: PCL"],
            yticklabels=["True: No PCL", "True: PCL"])
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

In [ ]:
test_df_eval = test_df.copy()
test_df_eval["pred"]  = test_preds
test_df_eval["prob"]  = test_probs
test_df_eval["error"] = test_df_eval["label"] != test_df_eval["pred"]

# False positives: predicted PCL, actually not
fp = test_df_eval[(test_df_eval["pred"] == 1) & (test_df_eval["label"] == 0)]
# False negatives: missed PCL
fn = test_df_eval[(test_df_eval["pred"] == 0) & (test_df_eval["label"] == 1)]

print(f"False Positives: {len(fp)}  |  False Negatives: {len(fn)}")
print()
print("── 5 False Positives (predicted PCL, was No PCL) ──")
for _, row in fp.nlargest(5, "prob").iterrows():
    print(f"  [{row['prob']:.2f}] {row['text'][:120]}...")
    print()

print("── 5 False Negatives (missed PCL) ──")
for _, row in fn.nsmallest(5, "prob").iterrows():
    print(f"  [{row['prob']:.2f}] [{row['orig_label']}] {row['text'][:120]}...")
    print()

## 7. Generate Submission Files

`dev.txt` — one prediction per line for the official dev set (2,094 lines).  
`test.txt` — one prediction per line for the official unlabelled test set (3,832 lines).  

Both files go in the repo root for leaderboard submission.

In [ ]:
def predict_file(model, texts, threshold, batch_size=32):
    """Run inference on a list of texts and return binary predictions."""
    dataset = PCLDataset(texts)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            probs  = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)

    return (np.array(all_probs) >= threshold).astype(int)


# Predictions on official dev set (our test; we have labels to verify)
dev_preds = predict_file(model, test_df["input_text"].tolist(), BEST_THRESHOLD)
with open("../dev.txt", "w") as f:
    f.write("\n".join(map(str, dev_preds)) + "\n")
print(f"dev.txt written: {len(dev_preds)} lines  |  PCL predicted: {dev_preds.sum()}")

# Predictions on unlabelled official test set
submit_preds = predict_file(model, submit_df["input_text"].tolist(), BEST_THRESHOLD)
with open("../test.txt", "w") as f:
    f.write("\n".join(map(str, submit_preds)) + "\n")
print(f"test.txt written: {len(submit_preds)} lines  |  PCL predicted: {submit_preds.sum()}")

assert len(submit_preds) == 3832, "test.txt must have exactly 3,832 lines"

## 8. Ensemble (Optional)

Train the same model with different random seeds, then take a **majority vote**
across the 3 best runs. This typically adds 0.5–1.5 F1 at no architectural cost.
Run this cell only if you have time and GPU budget.

In [ ]:
SEEDS        = [42, 7, 123]
ENSEMBLE_DIR = "../BestModel/ensemble"
os.makedirs(ENSEMBLE_DIR, exist_ok=True)

ensemble_val_probs  = []
ensemble_test_probs = []

for seed in SEEDS:
    print(f"\n{'='*50}\nTraining with seed={seed}\n{'='*50}")
    set_seed(seed)

    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2,
        hidden_dropout_prob=DROPOUT,
        attention_probs_dropout_prob=DROPOUT,
        ignore_mismatched_sizes=True
    ).to(DEVICE)

    pg  = get_grouped_llrd_params(m, BASE_LR, LAMBDA, N_GROUPS)
    opt = AdamW(pg)
    sch = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)

    best_f1_seed, best_state_seed, patience_ctr_seed = 0.0, None, 0

    for epoch in range(1, N_EPOCHS + 1):
        m.train(); opt.zero_grad(); ep_loss = 0
        for step, batch in enumerate(train_loader):
            labels = batch.pop("labels").to(DEVICE)
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            loss   = m(**batch, labels=labels).loss / GRAD_ACCUM
            loss.backward(); ep_loss += loss.item() * GRAD_ACCUM
            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad()

        _, _, vf1 = evaluate(m, val_loader)
        print(f"  Epoch {epoch}  loss={ep_loss/len(train_loader):.4f}  val_F1={vf1:.4f}")
        if vf1 > best_f1_seed:
            best_f1_seed = vf1
            best_state_seed = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            patience_ctr_seed = 0
        else:
            patience_ctr_seed += 1
            if patience_ctr_seed >= PATIENCE:
                print("  Early stop."); break

    m.load_state_dict(best_state_seed); m.to(DEVICE)
    m.save_pretrained(os.path.join(ENSEMBLE_DIR, f"seed{seed}"))

    vp, _, _ = evaluate(m, val_loader)
    ensemble_val_probs.append(vp)

    sp = predict_file(m, submit_df["input_text"].tolist(), threshold=0.5)
    ensemble_test_probs.append(sp)  # raw predictions for majority vote

# Majority vote across seeds
def majority_vote(preds_list, threshold=BEST_THRESHOLD):
    stacked = np.stack(preds_list, axis=0)   # shape: (n_seeds, n_samples)
    return (stacked.mean(axis=0) >= 0.5).astype(int)  # vote

ensemble_val_preds  = majority_vote([(p >= BEST_THRESHOLD).astype(int) for p in ensemble_val_probs])
ensemble_test_preds = majority_vote(ensemble_test_probs)

ens_f1 = f1_score(val_labels, ensemble_val_preds, pos_label=1)
print(f"\nEnsemble Val F1: {ens_f1:.4f}  (vs single model: {best_f1_at_t:.4f})")

with open("../dev.txt",  "w") as f: f.write("\n".join(map(str, predict_file(model, test_df["input_text"].tolist(), BEST_THRESHOLD))) + "\n")
with open("../test.txt", "w") as f: f.write("\n".join(map(str, ensemble_test_preds)) + "\n")
print("Ensemble dev.txt and test.txt written.")